# Sharp-curve experiments

This experimental controller extends the single-strip baseline. It has not been tested on your robot. Use an isolated black line on a white floor; it does not interpret intersections, green markers, or gaps. Each band selects its largest dark contour and can mistake shadows or other objects for the line.

**Run one cell at a time, never Run All.** First resolve the repeated power alarms with the lab. Stop/shut down the other notebook kernel before opening this camera. Keep the physical power switch accessible. Software timeouts cannot protect against a frozen process or driver.

## 1. Initialize once
Use a Jetson notebook kernel. Do not rerun camera creation in the same session.


In [ ]:
import time
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets

from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336)

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)

In [ ]:
latest = {"sample": None}

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")

## 2. Detect three strips
Copy your calibrated threshold below. In preview, verify all strips cover floor, and each green dot follows tape. Farther up the image is farther ahead only for the expected forward/downward camera mounting.


In [ ]:
BLACK_THRESHOLD = 70  # Copy your calibrated threshold here.
# Fractions of image height: change these if the far strip is above the floor.
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

def detect_curve(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    overlay = frame.copy()
    positions = {}
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            moments = cv2.moments(contour)
            # Permit a broad horizontal segment, but reject an almost dark strip.
            if 0.005 < fraction < 0.80 and moments["m00"] > 0:
                cx = moments["m10"] / moments["m00"]
                cy = int(moments["m01"] / moments["m00"]) + y0
                positions[name] = (cx - width / 2.0) / (width / 2.0)
                cv2.circle(overlay, (int(cx), cy), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
        cv2.putText(overlay, name, (3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    cv2.line(overlay, (width // 2, 0), (width // 2, height - 1), (0, 255, 255), 1)
    return positions, overlay, mask


## 3. Steering and bounded turns
Copy your measured minimum reliable speeds. A stopped inner wheel makes a tighter turn. After checking wheel directions, `ALLOW_REVERSE = True` permits an in-place pivot. These parameters require hardware tuning. The direction is latched during a turn and exits only when the near line and steering estimate are centred; a timeout stops unsuccessful turns. A missing line permits at most a short search in the last clear turn direction.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = False  # True enables an in-place turn with one wheel reversing.
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5

def clamp(value, low, high):
    return max(low, min(high, value))

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Require the near line and combined direction to agree before exiting.
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"


In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Stationary preview ready")
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_curves(seconds=10, drive=False):
    robot.stop()
    controller = CurveController()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]
            if sample is None:
                print("Stopped: no camera frame. Wait briefly, then retry.")
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print("Stopped: stale camera frame")
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, mask = detect_curve(frame)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print("Stopped: time limit or slow image processing")
                break
            left, right, state = controller.step(positions, now)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))
                status_view.value = "%s | %s | left %.2f right %.2f" % (
                    "DRIVING" if drive else "PREVIEW", state, left, right)
                next_display = now + 0.1
            if controller.stopped and drive:
                print(state)
                break
            if controller.stopped:
                # Preview remains usable while repositioning the stationary robot.
                controller = CurveController()
            time.sleep(0.01)
    finally:
        robot.stop()
    print("Run ended; motors stopped.")


## 4. Stationary preview
Move the robot by hand over the approach and bend. Watch the dots and proposed commands. Right turn: left command exceeds right. Left turn: right exceeds left. No motor commands other than stop are issued in preview.


In [ ]:
run_curves(seconds=20, drive=False)


## 5. Short driving test
After preview works, uncomment the next call to start a two-second test. Start before a broad curve, then test progressively tighter curves in both directions. Keep the robot on the floor with clear space. Change one parameter at a time.


In [ ]:
# Uncomment deliberately to move the robot:
# run_curves(seconds=2, drive=True)


## Tuning

- Line dots wrong: fix lighting, threshold, or band placement before changing steering.
- Gentle bends understeer: increase `KP` slightly; oscillation: reduce it.
- Tight turn enters too late: decrease `TURN_ENTER` gradually, e.g. 0.55 to 0.45.
- Wheel stalls: use the measured reliable motor command; do not keep lowering speed.
- Stopped-wheel turn is too wide: test `ALLOW_REVERSE = True` in a short run after verifying wheel directions.
- Turn timeout: inspect detections and pivot speed before increasing duration. Search is bounded deliberately; do not extend it blindly.
- A true right-angle corner can become invisible or look like a centred horizontal bar. This heuristic is not a complete right-angle/intersection detector. Save failure frames for a dedicated corner state.

## 6. Cleanup
Run when your experiments finish, not between each test. To reopen the camera afterward, restart the kernel and initialize once.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()